# Requirement
We have collected fire calls data file sf-fire-calls.csv

1. Read the data file.
2. Load the data into table for analysis.
3. Verify all 175296 records are loaded correctly.
4. The table is predefined as below.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dev.spark_db.sf_fire_calls (
    CallNumber INT,
    UnitID STRING,
    IncidentNumber INT,
    CallType STRING,
    CallDate DATE,
    WatchDate DATE,
    CallFinalDisposition STRING,
    AvailableDtTm TIMESTAMP,
    Address STRING,
    City STRING,
    Zipcode STRING,
    Battalion STRING,
    StationArea STRING,
    Box STRING,
    OriginalPriority STRING,
    Priority STRING,
    FinalPriority INT,
    ALSUnit BOOLEAN,
    CallTypeGroup STRING,
    NumAlarms INT,
    UnitType STRING,
    UnitSequenceInCallDispatch INT,
    FirePreventionDistrict STRING,
    SupervisorDistrict STRING,
    Neighborhood STRING,
    Location STRING,
    RowID STRING,
    Delay DOUBLE
);

# Solution Approach
1. Check the data file structure.
2. Read the data file and create a dataframe.
3. Check the record count.
4. Check the dataframe for potential problem.
5. Verify the datafram schem with the target table schema.
6. Transform the dataframe to match the target table structure (withColumns, to_date, to_timestamp, cast)
7. Save the final dataframe into target table
8. Verify the table count.

In [0]:
%sql
drop table if exists dev.spark_db.sf_fire_calls

In [0]:
# Step 1 and Step 2
'''
1. Check the data file structure.
2. Read the data file and create a dataframe.
'''
raw_fire_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("/Volumes/dev/spark_db/datasets/spark_programming/data/sf-fire-calls.csv")
)

In [0]:
# Step 3
'''Check the record count.'''

print(f"Total records read from CSV File: {raw_fire_df.count()}")

if(raw_fire_df.count() != 175296):
    raise Exception ("The number of records in the csv file is not correct.")

In [0]:
# Step 4
''' Check the datafram for potential problem'''
raw_fire_df.display()

In [0]:
#  Step 5
'''Verify the datafram schem with the target table schema.'''
raw_fire_df.printSchema()

In [0]:
# Step 6
'''Create a new dataframe with the correct column names'''
from pyspark.sql.functions import to_timestamp, expr

# fire_df = raw_fire_df.withColumn("AvailableDtTm", to_timestamp("AvailableDtTm", "MM/dd/yyyy hh:mm:ss a"))
fire_df = raw_fire_df.withColumns({
    "AvailableDtTm" : to_timestamp("AvailableDtTm", "MM/dd/yyyy hh:mm:ss a"),
    "Zipcode" : expr("cast(Zipcode as string)")
})
fire_df.printSchema()
fire_df.display()

In [0]:
# Step 7
'''Write the dataframe to the target table'''
fire_df.write.mode("overwrite").saveAsTable("dev.spark_db.sf_fire_calls")

In [0]:
# Step 8
'''Check the record count'''
spark.sql("select count(*) from dev.spark_db.sf_fire_calls").display()

In [0]:
%sql
select * from dev.spark_db.sf_fire_calls